# Interactive Growing Degree Day (GDD) Analysis

**Dataset:** Somerset, CA Hourly Temperature Data (selectable year: 2022-2025)

This notebook provides an interactive tool to explore Growing Degree Days (GDD) using hourly temperature data. Adjust the parameters below to see how base temperature, saturation temperature, year selection, and date range affect GDD calculations.

## What is GDD?

Growing Degree Days measure heat accumulation useful for predicting crop development. The calculation works as follows:

**Method 1 (hourly):**
```
contribution = max(0, min(hourly_temp, saturation) - base)
daily_gdd = sum(hourly_contributions) / hours_with_data
```

**Method 2 (old/simple):**
```
daily_gdd_old = max(0, 0.5 * (max_temp + min_temp) - base)
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Year options
year_options = [2022, 2023, 2024, 2025]

# Create year dropdown first
year_dropdown = widgets.Dropdown(
    options=year_options,
    value=2024,
    description="Year:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="150px")
)

# Function to load data for a given year
def load_year_data(year):
    return pd.read_csv(f"/workspaces/ag-skills-demo/data/growing_degree_days/somerset_{year}.csv")

# Load initial year data
df = load_year_data(2024)
df["datetime"] = pd.to_datetime(df["datetime"])
df["date"] = df["datetime"].dt.date

# Get available dates for the dropdown
available_dates = sorted(df["date"].unique())
date_options = [str(d) for d in available_dates]

print(f"Loaded {len(df)} hourly records for 2024")
print(f"Date range: {available_dates[0]} to {available_dates[-1]}")

Loaded 8784 hourly records for 2024
Date range: 2024-01-01 to 2024-12-31


In [23]:
# Create interactive widgets
base_slider = widgets.FloatSlider(
    value=10.0,
    min=0.0,
    max=20.0,
    step=0.5,
    description="Base (C):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

saturation_slider = widgets.FloatSlider(
    value=30.0,
    min=15.0,
    max=40.0,
    step=0.5,
    description="Saturation (C):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

start_date_dropdown = widgets.Dropdown(
    options=date_options,
    value="2024-04-01",
    description="Start Date:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="200px")
)

end_date_dropdown = widgets.Dropdown(
    options=date_options,
    value="2024-11-15",
    description="End Date:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="200px")
)

show_gdd_old_checkbox = widgets.Checkbox(
    value=False,
    description="Show GDD_old (daily min/max)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

# Arrange widgets in a box
widgets_box = widgets.VBox([
    widgets.HTML("<h3>Adjust GDD Parameters</h3>"),
    widgets.HTML("<hr>"),
    year_dropdown,
    base_slider,
    saturation_slider,
    start_date_dropdown,
    end_date_dropdown,
    show_gdd_old_checkbox,
])

In [24]:
def calculate_gdd(df, base, saturation, start_date, end_date):
    """
    Calculate Growing Degree Days from hourly temperature data.
    """
    # Filter to date range
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    df_filtered = df[(df["datetime"] >= start_dt) & (df["datetime"] <= end_dt + pd.Timedelta(days=1))].copy()
    
    if len(df_filtered) == 0:
        return pd.DataFrame(columns=["date", "gdd", "cumulative_gdd", "gdd_old", "cumulative_gdd_old"])
    
    df_filtered = df_filtered.dropna(subset=["temperature_c"])
    
    # Calculate hourly contribution
    df_filtered["capped_temp"] = df_filtered["temperature_c"].clip(upper=saturation)
    df_filtered["contribution"] = (df_filtered["capped_temp"] - base).clip(lower=0)
    
    # Group by date and calculate daily GDD
    daily = df_filtered.groupby("date").agg(
        gdd_sum=("contribution", "sum"),
        hour_count=("contribution", "count"),
        t_min=("temperature_c", "min"),
        t_max=("temperature_c", "max")
    ).reset_index()
    
    daily["gdd"] = daily["gdd_sum"] / daily["hour_count"]
    daily["cumulative_gdd"] = daily["gdd"].cumsum()
    
    # GDD_old: max(0, 0.5 * (t_max + t_min) - base)
    daily["gdd_old"] = (0.5 * (daily["t_max"] + daily["t_min"]) - base).clip(lower=0)
    daily["cumulative_gdd_old"] = daily["gdd_old"].cumsum()
    
    return daily[["date", "gdd", "cumulative_gdd", "gdd_old", "cumulative_gdd_old"]]

In [25]:
def update_plot(year, base, saturation, start_date, end_date, show_gdd_old):
    """
    Update the GDD plot based on current parameter values.
    """
    clear_output(wait=True)
    
    # Display widgets at top
    display(widgets_box)
    
    # Load data for the selected year
    df = load_year_data(year)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["date"] = df["datetime"].dt.date
    
    # Update date dropdown options for this year
    available_dates = sorted(df["date"].unique())
    date_options = [str(d) for d in available_dates]
    start_date_dropdown.options = date_options
    end_date_dropdown.options = date_options
    
    # Validate start/end dates are in range, default to Apr 1 and Nov 15 if available
    if start_date not in date_options:
        start_date = date_options[0]
    if end_date not in date_options:
        end_date = date_options[-1]
    
    # Set sensible defaults for start/end dates if they're in the options
    default_start = f"{year}-04-01"
    default_end = f"{year}-11-15"
    if default_start in date_options and default_end in date_options:
        start_date_dropdown.value = default_start
        end_date_dropdown.value = default_end
        start_date = default_start
        end_date = default_end
    
    # Calculate GDD
    gdd_data = calculate_gdd(df, base, saturation, start_date, end_date)
    
    if len(gdd_data) == 0:
        print("No data available for the selected parameters.")
        return
    
    # Create figure with dual y-axis
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Convert dates for plotting
    dates = pd.to_datetime(gdd_data["date"])
    
    # Daily GDD as bars - use matplotlib date numbers for positioning
    color_daily = "#3498db"
    color_daily_old = "#e67e22"
    width = 0.4
    
    # Convert dates to matplotlib date format (floats)
    dates_num = plt.matplotlib.dates.date2num(dates)
    
    if show_gdd_old:
        ax1.bar(dates_num - width/2, gdd_data["gdd"], width=width, color=color_daily, alpha=0.6, label="Daily GDD (hourly)")
        ax1.bar(dates_num + width/2, gdd_data["gdd_old"], width=width, color=color_daily_old, alpha=0.6, label="Daily GDD_old (min/max)")
    else:
        ax1.bar(dates_num, gdd_data["gdd"], color=color_daily, alpha=0.6, label="Daily GDD", width=0.8)
    
    ax1.set_xlabel("Date", fontsize=11)
    ax1.set_ylabel("Daily GDD (C)", fontsize=11)
    ax1.tick_params(axis="y")
    ax1.set_ylim(bottom=0)
    
    # Cumulative GDD as line on secondary axis
    ax2 = ax1.twinx()
    color_cumulative = "#e74c3c"
    color_cumulative_old = "#27ae60"
    
    ax2.plot(dates, gdd_data["cumulative_gdd"], color=color_cumulative, linewidth=2.5, label="Cumulative GDD (hourly)")
    
    if show_gdd_old:
        ax2.plot(dates, gdd_data["cumulative_gdd_old"], color=color_cumulative_old, linewidth=2.5, linestyle="--", label="Cumulative GDD_old (min/max)")
    
    ax2.set_ylabel("Cumulative GDD (C)", fontsize=11)
    ax2.tick_params(axis="y")
    ax2.set_ylim(bottom=0)
    
    # Title with parameters
    plt.title(f"Growing Degree Days Analysis\nYear: {year} | Base: {base}C | Saturation: {saturation}C | Range: {start_date} to {end_date}", 
              fontsize=13, fontweight="bold")
    
    # Add legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    
    # Add grid
    ax1.grid(True, alpha=0.3, axis="y")
    
    # Rotate x-axis labels
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    
    # Print summary statistics
    final_cumulative = gdd_data["cumulative_gdd"].iloc[-1]
    avg_daily = gdd_data["gdd"].mean()
    max_daily = gdd_data["gdd"].max()
    max_date = gdd_data.loc[gdd_data["gdd"].idxmax(), "date"]
    
    print(f"\n{'='*60}")
    print(f"Summary (Year: {year}, Base: {base}C, Sat: {saturation}C)")
    print(f"Date Range: {start_date} to {end_date}")
    print(f"{'='*60}")
    print(f"Days analyzed:           {len(gdd_data)}")
    print(f"Final cumulative GDD:   {final_cumulative:.2f}C (hourly method)")
    print(f"Average daily GDD:      {avg_daily:.2f}C")
    print(f"Max daily GDD:           {max_daily:.2f}C (on {max_date})")
    
    if show_gdd_old:
        final_cumulative_old = gdd_data["cumulative_gdd_old"].iloc[-1]
        print(f"Final cumulative GDD_old: {final_cumulative_old:.2f}C (min/max method)")
        print(f"Difference (GDD - GDD_old): {final_cumulative - final_cumulative_old:.2f}C")
    
    # ============================================================
    # Additional plots: GDD_hourly vs GDD_old comparison
    # ============================================================
    
    # Create figure with 2 subplots below
    fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Daily GDD comparison (scatter)
    ax3 = axes[0]
    ax3.scatter(gdd_data["gdd"], gdd_data["gdd_old"], alpha=0.6, color="#9b59b6", edgecolors="black", linewidth=0.5)
    # Add diagonal line (perfect correlation)
    max_val = max(gdd_data["gdd"].max(), gdd_data["gdd_old"].max())
    ax3.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label="1:1 line")
    ax3.set_xlabel("GDD (hourly method)", fontsize=11)
    ax3.set_ylabel("GDD (min/max method)", fontsize=11)
    ax3.set_title("Daily GDD: Hourly vs Min/Max", fontsize=12, fontweight="bold")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(left=0)
    ax3.set_ylim(bottom=0)
    
    # Plot 2: Cumulative GDD comparison (scatter)
    ax4 = axes[1]
    ax4.scatter(gdd_data["cumulative_gdd"], gdd_data["cumulative_gdd_old"], alpha=0.6, color="#1abc9c", edgecolors="black", linewidth=0.5)
    max_cum = max(gdd_data["cumulative_gdd"].max(), gdd_data["cumulative_gdd_old"].max())
    ax4.plot([0, max_cum], [0, max_cum], 'r--', alpha=0.5, label="1:1 line")
    ax4.set_xlabel("Cumulative GDD (hourly method)", fontsize=11)
    ax4.set_ylabel("Cumulative GDD (min/max method)", fontsize=11)
    ax4.set_title("Cumulative GDD: Hourly vs Min/Max", fontsize=12, fontweight="bold")
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(left=0)
    ax4.set_ylim(bottom=0)
    
    fig2.tight_layout()
    plt.show()

In [26]:
# Create interactive output and display
output = widgets.interactive_output(
    update_plot,
    {
        "year": year_dropdown,
        "base": base_slider,
        "saturation": saturation_slider,
        "start_date": start_date_dropdown,
        "end_date": end_date_dropdown,
        "show_gdd_old": show_gdd_old_checkbox
    }
)

display(output)

Output()

## How to Use This Notebook

1. **Select Year** - Choose which year's hourly temperature data to analyze (2022-2025)
2. **Adjust the Base Temperature** - The minimum temperature threshold for growth. Crops don't grow below this temperature.
3. **Adjust the Saturation Temperature** - The maximum effective temperature. Above this, temperature doesn't contribute to additional growth (growth plateaus).
4. **Change the Start Date** - When to begin accumulating GDD. Useful for simulating different planting dates.
5. **Change the End Date** - When to stop accumulating GDD. Useful for defining the growing season.
6. **Show GDD_old (daily min/max)** - Toggle to overlay the simple method GDD calculation: `max(0, 0.5 * (T_max + T_min) - base)`.
7. **Observe the Results**:
   - **Blue bars** = Daily GDD (hourly method)
   - **Orange bars** = Daily GDD_old (min/max method, when enabled)
   - **Red line** = Cumulative GDD (hourly method)
   - **Green dashed line** = Cumulative GDD_old (min/max method, when enabled)
   - **Scatter plots** = Compare hourly vs min/max methods directly

### Example Interpretations

| Scenario | Interpretation |
|----------|----------------|
| Lower base (5C) | More hours count toward GDD, higher cumulative total |
| Lower saturation (25C) | Capping temps earlier reduces daily contributions |
| Later start date | Fewer days in accumulation period, lower cumulative GDD |
| GDD vs GDD_old | Compare hourly-integrated vs simple average methods |